### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vector store
## from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
##utilities
import numpy as np
from typing import List, Tuple

In [ ]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")

### 1. Sample Data

In [ ]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


In [ ]:
##save the documents to text files
import tempfile
temp_dir=tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)
print(f"Documents saved to: {temp_dir}")

### 2. Document Loading

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
#Load documents from the temporary directory
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls= TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

In [ ]:
documents =loader.load()
print(f"Loaded {len(documents)} documents.")
print("Sample Document:")
print(documents[0].page_content[:500])

### Document Splitting

In [ ]:
# Intialize text splitter
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function= len,
    separators=[" "]
)
chunks= text_splitter.split_documents(documents)
print(f"Total Chunks Created: {len(chunks)} from {len(documents)} documents.")
print("Sample Chunk:")
print(chunks[0].page_content[:500])
print(f"Chunk 1 Metadata: {chunks[0].metadata}")
print(chunks[1].page_content[:500])
print(f"Chunk 2 Metadata: {chunks[1].metadata}")


### Embedding Models

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

In [ ]:
sample_query="Machine learning"
vector = embeddings.embed_query(sample_query)
vector

### INitialize the ChromaDB Store and Store the chunks in Vector Representation

In [ ]:
## Create a chromadb directory
# make a directory for chromadb if it doesn't exist
if not os.path.exists("./chromadb"):
    os.makedirs("./chromadb")

persist_dir ="./chromadb"
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
    collection_name="chroma_documents"
)
##vectorstore.add_documents(chunks)
print(f"Created {vectorstore._collection.count()} vectors")
print(f"Stored at {persist_dir}")

### Similarity Search

In [ ]:
sample_query="What is NLP"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

In [ ]:
sample_query="What is deep learning"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

In [ ]:
print(f"Sample Query: {sample_query}")
print(f"Top {len(similar_docs)} Similar Chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\nChunk {i+1} Metadata: {doc.metadata}")
    print(f"Chunk {i+1} Content: {doc.page_content[:500]}...")

In [ ]:
results_scores = vectorstore.similarity_search_with_score(sample_query,k=3)
results_scores

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG System

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatOpenAI(
    model_name="gpt-4.1",
    temperature=0
)

In [ ]:
test_response= llm.invoke("What is deep learning")
print(test_response)

In [ ]:
from langchain.chat_models.base import init_chat_model
llm = init_chat_model(
    model="gpt-4o",
    model_provider="openai"
)
test_response= llm.invoke("What is deep learning")
print(test_response)

### Modern RAG Chain

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
# import combine documents
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [ ]:
# convert vectore store to retriever
retrieve = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
retrieve

In [ ]:
#create prompt template
system_prompt = """
You are an assistant fir quaestion answering tasks.
Use the following pieces of retrieval context to anwser the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:{context}
"""

prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": system_prompt},
    {"role":"human", "content":"{input}"}
])

In [ ]:
prompt

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.


#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [ ]:
#create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

This chain:

- This creates a chain that takes retrieved documents.
- It inserts their text into your prompt’s {context} placeholder.
- Then it sends the full prompt (context + user question) to llm.
- Output is the model’s answer text.

In [ ]:
### Create final Rag chain
rag_chain = create_retrieval_chain(retrieve, document_chain)
rag_chain

This wraps two steps together:
- Step A: retrieve finds relevant chunks from your vector store.
- Step B: document_chain uses those chunks to generate an answer.
- So now you have one callable chain that does retrieval + generation end-to-end.

1. First chain (create_stuff_documents_chain)
 - It combines LLM + prompt template.
 - It expects context and input.
 - It does not retrieve documents itself.
 - It “maps” retrieved docs into {context} and asks the LLM to answer.

2. Second chain (create_retrieval_chain)
 - It combines retriever + first chain.
 - At run time (invoke), it retrieves relevant docs from the vector store.
 - Then it passes those docs as context into the first chain.
So you get one end-to-end RAG pipeline: question -> retrieve context -> generate answer.
So your core idea is right: first chain is answer-generation with context slot, second chain adds retrieval so that slot gets real relevant documents automatically.

In [ ]:
response = rag_chain.invoke({"input": "What is deep learning?"})
response


In [ ]:
def query_rag(query:str):
    print(f"Query: {query}")
    print("-"*50)

    response = rag_chain.invoke({"input": query})
    print(f"Response: {response['answer']}")

    for i, doc in enumerate(response["context"]):
        print(f"\n Source {i+1} Metadata: {doc.metadata}")
        print(f" Source {i+1} Content: {doc.page_content[:500]}...")

    return response

In [ ]:
query_rag("What machine learning is?")

### Create RAG Chain Alternative - Using LCEL(LangChain Expression Language)

In [ ]:
#creating RAG chain without using utility functions
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSerializable, RunnableParallel

In [ ]:
#Creating a a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""
Use the following to context to answer the question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Provide specific details from the context to support your answer.

Context: {context}
Question: {question}
Answer:
"""
)
custom_prompt

In [ ]:
#format the output document for the custom prompt
def format_docs(docs):
    return "\n\n".join([f"Source: {doc.metadata['source']}\nContent: {doc.page_content}" for doc in docs])


In [ ]:
#Build chain using LCEL
rag_chain_lcel = (
    {
        "context": retrieve | format_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt 
    | llm 
    | StrOutputParser()
)
rag_chain_lcel

In [ ]:
response = rag_chain_lcel.invoke("What is deep learning?")
print(f"Response: {response}")

In [ ]:
format_docs(retrieve.invoke("What is deep learning?"))

### Add documents to the existing vector store

In [ ]:
vectorstore

In [ ]:
# vector store with new documents
new_document = """
Generative Artificial Intelligence (AI)

Generative AI refers to a class of artificial intelligence models that can generate new content, such as text, 
images, or audio, based on patterns learned from existing data. These models use techniques 
like deep learning and transformers to create outputs that are often indistinguishable from human-generated content. Generative AI has applications in various fields, including creative writing, art generation, and even code synthesis.

Generative AI models, such as GPT-3 and DALL-E, have demonstrated remarkable capabilities in generating 
coherent text and realistic images. These models are trained on vast amounts of data and can produce 
creative outputs that mimic human style and creativity. However, they also raise ethical concerns 
regarding misinformation, bias, and the potential for misuse.

Generative AI is a rapidly evolving field that has expanded beyond traditional text generation to include a 
wide range of applications. With the advent of powerful models and techniques, generative AI can now create not only text but also images, 
videos, music, code, and even complex documents. This has opened up new possibilities for creativity and 
productivity across various domains. Generative AI can be used to schedule events, connect with other tools, 
and even create agentic AI that can perform tasks autonomously. As the technology continues to advance, 
we can expect to see even more innovative applications of generative AI in the future.


As the field of generative AI continues to evolve, it is crucial to balance innovation with ethical 
considerations. Responsible development and deployment of generative AI can lead to incredible 
advancements while minimizing potential risks. By fostering collaboration between researchers, 
policymakers, and industry leaders, we can ensure that generative AI is used for the betterment 
of society while addressing concerns related to misinformation, bias, and misuse. The future of 
generative AI holds immense promise, and with careful stewardship, it can be a powerful tool 
for creativity, productivity, and positive impact across various domains.
"""

In [ ]:
new_document


In [ ]:
new_doc = Document(
    page_content=new_document,
    metadata={
        "source": "new_doc.txt",
        "topic": "Generative AI",
        "author": "John Doe"
    }
)
new_doc

### Add new document to vector store

#### split_document vs split_text

goal: Explain split_documents vs split_text for text splitters, in Markdown with examples.
What They Do

 - split_documents: Accepts a list of Document objects and returns a list of chunked Document objects (keeps metadata).
 - split_text: Accepts a single raw str and returns a list of text chunks (strings), no document metadata.

#### Key Differences

 - Input type: split_documents → list of Document; split_text → str.
 - Output type: split_documents → list of Document (with page_content + metadata); split_text → list of str.
 - Metadata: split_documents preserves and can attach/propagate metadata; split_text loses metadata.
 - Use-case: split_documents for pipelines where source attribution matters (citations, source fields).     split_text for quick ad-hoc splitting of plain strings.

In [ ]:
## Splitting the document into chunks
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

In [ ]:
## Add new document to vector store
vectorstore.add_documents(new_chunks)

In [ ]:
print(f"New Chunks {len(new_chunks)}. Total Vectors after adding new document: {vectorstore._collection.count()}")

In [ ]:
## query with the updated vector store
new_question = "What kind of things we can do with generative AI?"
result = query_rag(new_question)
result

### Advanced RAG Concepts
 - Advanced rag, conversational memory,the previous rag that we built dont have any memory of the previous interactions. 
 - We can add a conversational memory to the RAG chain to make it more interactive and context-aware. 
 - This way, the model can remember previous questions and answers, allowing for a more natural conversation flow.

 Example:
 - Q1 - We asked what is gnerative ai, 
 - Q2 - We asked what kind of things we can do with generative ai,
 - so the system should be able to remember the previous conversation and provide a more detailed answer based on the new document we added to the vector store.

#### Conversational Memory
- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HHumanMessage/AIMessage: Structure message types for conversation history

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
## create a prompt with conversation history
contextual_system_prompt ="""
Given a chat history and the latest question which might refer to previous conversation, 
formulate a standalone question which can be answered without the need of chat history.
DO NOT answer the question, just reformulate it if needed or return the question as is if it 
doesn't refer to previous conversation.
"""
# reframing the question to be standalone question to include the previous conversation.
#  This will help the retriever to retrieve relevant documents based on the standalone question.
contextual_prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": contextual_system_prompt},
    MessagesPlaceholder(variable_name="chat_history"),
    {"role":"user", "content": "{input}"} 
    # input is the latest question from the user which might refer to previous conversation, 
    # we will reframe it to be standalone question to include the previous conversation.
])

In [ ]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retrieve, contextual_prompt
)
history_aware_retriever

In [ ]:
# Create a new document chain with history
qa_system_prompt ="""
You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum to answer the question and be concise.

Context: {context} 
"""
# this context will include the retrieved documents based on the 
# standalone question which includes the previous conversation.

qa_prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": qa_system_prompt},
    {"role":"user", "content": "{input}"}
])

In [ ]:
# Create a new document chain with the custom QA prompt
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)
question_answer_chain

In [ ]:
# Create a conversation RAG chain
conversation_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
conversation_rag_chain

In [ ]:
chat_history = []
result1 = conversation_rag_chain.invoke({
    "input": "What is generative AI?", 
    "chat_history": chat_history
})
chat_history.extend(
    [
        HumanMessage(content="What is generative AI?"),
        AIMessage(content=result1["answer"])
    ]
)
print(f"Question: What is generative AI?")
print(f"Answer: {result1['answer']}")

In [ ]:
## Follow up question
result2 = conversation_rag_chain.invoke({
    "input": "What are the applications of generative AI?", 
    "chat_history": chat_history
})
print(f"\nQuestion: What are its applications?")
print(f"Answer: {result2['answer']}")

### Complete Flow Diagram
FIRST QUESTION

User
  │
  ▼
Tell me about Virat Kohli
  │
  ▼
History Aware Retriever
(chat history empty)
  │
  ▼
Tell me about Virat Kohli
  │
  ▼
Vector DB Search
  │
  ▼
Relevant Documents
  │
  ▼
QA Prompt
  │
  ▼
Final Answer


FOLLOW-UP QUESTION

User
  │
  ▼
Which team does he play for?
  │
  ▼
History Aware Retriever
  │
  ▼
Chat History +
Current Question
  │
  ▼
Rewritten Question:
Which team does Virat Kohli play for?
  │
  ▼
Vector DB Search
  │
  ▼
Relevant Documents
  │
  ▼
QA Prompt
  │
  ▼
Answer:
Virat Kohli plays for RCB.


#### LLM Call #1

Inside:
- create_history_aware_retriever(...)
- Purpose:
  Rewrite the question
- Example:
  he → Virat Kohli
  it → Oracle
  that company → Microsoft

#### LLM Call #2

Inside:
- create_stuff_documents_chain(...)
- Purpose:
  Answer the question
- using:
  Question
  +
  Retrieved Documents

### GROQ LLM

In [95]:
load_dotenv()
os.environ["1-Chromadb.ipynb"] = os.getenv("GROQ_API_KEY")

In [97]:
from langchain_groq import ChatGroq
from langchain.chat_models.base import init_chat_model

langchain_groq1= ChatGroq(model="qwen/qwen3-32b")
langchain_groq1.invoke("What is generative AI?")

AIMessage(content='<think>\nOkay, the user is asking what generative AI is. Let me start by breaking down the term. Generative AI is a subset of artificial intelligence, so I should explain that it\'s focused on generating new content rather than just analyzing existing data. \n\nFirst, I need to define it clearly. Maybe mention that it creates content like text, images, music, etc. Then, I should explain how it works, like using neural networks and machine learning models trained on large datasets. Terms like GANs (Generative Adversarial Networks) and transformers might come up here. I should simplify these concepts without getting too technical.\n\nNext, examples would help. If I list some applications, like text generation (e.g., ChatGPT, GPT-4), image creation (DALL-E, MidJourney), or music composition, the user can visualize its uses better. Also, mentioning different industries where it\'s applied—education, healthcare, marketing—could show its versatility.\n\nI should also touch

In [98]:
langchain_groq2 = init_chat_model(
    model="qwen/qwen3-32b",
    model_provider="groq"
)
langchain_groq2.invoke("What is LLM?")

AIMessage(content='<think>\nOkay, the user is asking what LLM stands for. First, I need to confirm if they\'re referring to Large Language Models. Since they mentioned "LLM" directly, it\'s likely they\'re asking about that. But I should also consider other possible meanings, like local loop management or other specialized terms in different fields. However, in the context of current technology discussions, LLM most commonly refers to Large Language Models.\n\nNext, I should explain what a Large Language Model is. Start with the basic definition: an AI model trained on vast amounts of text data to understand and generate human-like text. Mention key capabilities like answering questions, writing stories, emails, code, and more. Highlight the training process, using examples like GPT, BERT, and LLaMA.\n\nThen, discuss the applications. Users might be interested in how these models are used in real life, so list areas like customer service, content creation, education, and research. Also

### Nvidia AI EndPoints

In [ ]:
load_dotenv()
os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

In [10]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os
langchain_nvidia = ChatNVIDIA(
model="meta/llama-3.2-1b-instruct"
)
langchain_nvidia.invoke("Who is Virat Kohli?")

KeyboardInterrupt: 